In [ ]:
!pip install ultralytics roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="A828xM1sAEtjHMh5WHGh")
project = rf.workspace("ashishs-workspace-nho2b").project("traffic-detection-yc4eg")
version = project.version(1)
dataset = version.download("yolov8")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data=dataset.location + "/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device="0"
)

In [ ]:
from google.colab import files
files.download('runs/detect/train/weights/best.pt')

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt")
metrics = model.val()
print(metrics)

In [ ]:
from google.colab import files

files.download("runs/detect/train/BoxPR_curve.png")
files.download("runs/detect/train/BoxF1_curve.png")
files.download("runs/detect/train/confusion_matrix.png")
files.download("runs/detect/train/results.png")

In [ ]:
!ls

In [ ]:
import time

start = time.time()
results = model("https://ultralytics.com/images/bus.jpg")
end = time.time()

print("Inference Time:", end-start)
print("FPS:", 1/(end-start))

In [ ]:
!ls runs/detect/train

In [ ]:
model.predict(source="https://ultralytics.com/images/bus.jpg", save=True)

In [ ]:
!ls runs/detect

In [ ]:
from google.colab import files

files.download("runs/detect/train/results.csv")

In [ ]:
!pip install opencv-python

In [ ]:
!pip install mediapipe

In [ ]:
!pip install ultralytics --upgrade

In [ ]:
!pip install ultralytics --upgrade
!pip install opencv-python
!pip install pillow
!pip install matplotlib

In [ ]:
!ls runs/detect/train/weights

In [ ]:
from ultralytics import YOLO
from PIL import Image
from google.colab import files

model = YOLO("runs/detect/train/weights/best.pt")

uploaded = files.upload()

for fn in uploaded.keys():
    image = Image.open(fn)
    results = model(image)

    for r in results:
        r.show()
        r.save(filename="car.jpg")

In [ ]:
import cv2
from IPython.display import Video

video_path = "/content/Traffic.mp4"
cap = cv2.VideoCapture(video_path)

# Get original properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# 🔥 IMPORTANT: Different output file name
output_path = "/content/output_Traffic.mp4"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    frame_out = results[0].plot()
    out.write(frame_out)

cap.release()
out.release()

print("Processing Done ✅")

# Display video
Video(output_path, embed=True)

In [ ]:
import os
import time
from google.colab import files

# Add a small delay to ensure the file is fully written and closed
time.sleep(2) # Wait for 2 seconds

if os.path.exists(output_path):
    print(f"File '{output_path}' found. Attempting download.")
    files.download(output_path)
else:
    print(f"Error: File '{output_path}' still not found after waiting. Please verify the video creation process.")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
video_path = "/content/Traffic.mp4"
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

output_path = "/content/traffic_analysis.mp4"

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# Line position (middle of road)
line_y = height // 2

# Counters
vehicle_count = 0
class_count = defaultdict(int)
frame_vehicle_data = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]
    boxes = results.boxes

    current_frame_count = 0

    for box in boxes:
        cls = int(box.cls[0])
        label = model.names[cls]

        # Only count vehicles
        if label in ["car", "truck", "bus", "motorbike"]:
            current_frame_count += 1
            class_count[label] += 1

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            center_y = (y1 + y2) // 2

            # Line crossing check
            if abs(center_y - line_y) < 5:
                vehicle_count += 1

    frame_vehicle_data.append(current_frame_count)

    # Draw line
    cv2.line(frame, (0, line_y), (width, line_y), (0, 0, 255), 3)

    # Show counter
    cv2.putText(frame, f"Line Count: {vehicle_count}", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 3)

    annotated = results.plot()
    out.write(annotated)

cap.release()
out.release()

print("Processing Done ✅")

In [ ]:
print("Total Vehicle Summary:")
for k, v in class_count.items():
    print(f"{k} : {v}")

In [ ]:

print(f"Data Length: {len(frame_vehicle_data)}")
print(f"Data Samples: {frame_vehicle_data[:10]}")

In [ ]:
plt.figure()
plt.plot(frame_vehicle_data)
plt.xlabel("Frame Number")
plt.ylabel("Vehicle Count")
plt.title("Frame vs Vehicle Count")
plt.show()

In [ ]:
print("Total Frames Processed:", len(frame_vehicle_data))
print("First 10 Values:", frame_vehicle_data[:10])

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

model = YOLO("yolov8n.pt")

video_path = "/content/Traffic.mp4"
cap = cv2.VideoCapture(video_path)

frame_vehicle_data = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    count_this_frame = 0

    for box in results.boxes:
        cls = int(box.cls[0])
        label = model.names[cls]

        if label in ["car", "truck", "bus", "motorbike"]:
            count_this_frame += 1

    frame_vehicle_data.append(count_this_frame)

cap.release()

print("Frames processed:", len(frame_vehicle_data))
print("Sample counts:", frame_vehicle_data[:20])

In [ ]:
plt.figure()
plt.plot(frame_vehicle_data)
plt.xlabel("Frame Number")
plt.ylabel("Vehicle Count")
plt.title("Frame vs Vehicle Count")
plt.show()

In [ ]:
avg_density = np.mean(frame_vehicle_data)

print("Average Vehicles per Frame:", round(avg_density,2))

if avg_density < 5:
    print("Traffic: Low")
elif avg_density < 15:
    print("Traffic: Medium")
else:
    print("Traffic: High")

In [ ]:
avg_density = np.mean(frame_vehicle_data)

print("Average Vehicles per Frame:", round(avg_density,2))

if avg_density < 5:
    print("Traffic: Low")
elif avg_density < 15:
    print("Traffic: Medium")
else:
    print("Traffic: High")

In [ ]:
import pandas as pd

df = pd.read_csv("runs/detect/train/results.csv")
print(df.columns)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load CSV
df = pd.read_csv("runs/detect/train/results.csv")

# Plot mAP graph
plt.figure(figsize=(10,5))
plt.plot(df['metrics/mAP50(B)'], label='mAP@0.5')
plt.plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')

plt.xlabel("Epochs")
plt.ylabel("mAP")
plt.title("Training mAP Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['metrics/precision(B)'], label='Precision')
plt.plot(df['metrics/recall(B)'], label='Recall')

plt.xlabel("Epochs")
plt.ylabel("Value")
plt.title("Precision & Recall Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['train/box_loss'], label='Train Box Loss')
plt.plot(df['val/box_loss'], label='Val Box Loss')

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from IPython.display import Image
Image("/content/runs/detect/train/BoxPR_curve.png")

In [ ]:
from IPython.display import Image

Image("runs/detect/train/confusion_matrix.png")

In [ ]:
!pip install reportlab

import pandas as pd
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet

# Load CSV
df = pd.read_csv("runs/detect/train/results.csv")

final_map50 = df['metrics/mAP50(B)'].iloc[-1]
final_map5095 = df['metrics/mAP50-95(B)'].iloc[-1]
final_precision = df['metrics/precision(B)'].iloc[-1]
final_recall = df['metrics/recall(B)'].iloc[-1]

pdf_path = "/content/YOLO_Training_Report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4)
elements = []

styles = getSampleStyleSheet()
elements.append(Paragraph("<b>YOLOv8 Training Report</b>", styles['Title']))
elements.append(Spacer(1, 0.5 * inch))

elements.append(Paragraph(f"Final mAP@0.5: {round(final_map50,4)}", styles['Normal']))
elements.append(Paragraph(f"Final mAP@0.5:0.95: {round(final_map5095,4)}", styles['Normal']))
elements.append(Paragraph(f"Final Precision: {round(final_precision,4)}", styles['Normal']))
elements.append(Paragraph(f"Final Recall: {round(final_recall,4)}", styles['Normal']))

elements.append(Spacer(1, 0.5 * inch))

# Add training result image
elements.append(Paragraph("<b>Training Results Graph</b>", styles['Heading2']))
elements.append(RLImage("runs/detect/train/results.png", width=5*inch, height=3*inch))

elements.append(Spacer(1, 0.5 * inch))

elements.append(Paragraph("<b>Confusion Matrix</b>", styles['Heading2']))
elements.append(RLImage("runs/detect/train/confusion_matrix.png", width=5*inch, height=3*inch))

doc.build(elements)

print("PDF Report Generated Successfully ✅")

In [ ]:
from google.colab import files
files.download("/content/YOLO_Training_Report.pdf")

In [ ]:
files.download('runs/detect/train/weights/best.pt')

In [ ]:
pip install ultralytics torch

In [ ]:
!python src/train.py

In [ ]:
!ls

In [ ]:
!ls -R

In [ ]:
!git clone https://github.com/Ashu1631/Yolo_V8.git

In [ ]:
%cd Yolo_V8

In [ ]:
!ls

In [ ]:
!ls src

In [ ]:
!pip install ultralytics torch

In [ ]:
!python src/train.py

In [ ]:
!ls

In [ ]:
!git pull

In [ ]:
!ls

In [ ]:
!python src/train.py

In [ ]:
!pip install roboflow

In [ ]:
%cd /content
!rm -rf Yolo_V8
!git clone https://github.com/Ashu1631/Yolo_V8.git
%cd Yolo_V8
!pip install ultralytics roboflow
!python src/train.py

In [ ]:
!pip install supervision ultralytics

In [ ]:
import cv2
import supervision as sv
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import numpy as np

In [ ]:
model_custom = YOLO('best.pt')
model_custom = YOLO('yolov8n.pt')

In [ ]:
!pip install lapx